# 盈利质量因子回测

**因子逻辑**

- `CFO/NI`：经营现金流 / 净利润 → 高值表示盈利含金量高（不靠应收账款堆出来）
- `-Accruals/TA`：-(净利润 - 经营现金流) / 总资产 → 低应计项目 = 财务操纵空间小

两子因子截面 z-score 等权合成。**正向因子**：高值 → 预期收益更好。

**前视偏差处理**：季报数据 shift(1) 移到下一季度再使用（保守处理），
实际公告日比报告期末晚 1~3 个月，结论中需注明。

**回测步骤**
1. 数据准备（沪深 300 成分，2020-2024）
2. 单因子 IC 验证（Rank IC + ICIR）
3. 行业 + 市值中性化后复测
4. 五分位分层回测
5. 与现有英雄因子相关性检验

In [ ]:
import sys
sys.path.insert(0, '../../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'PingFang SC'
matplotlib.rcParams['axes.unicode_minus'] = False

from research.factors.earnings_quality.earnings_quality_factor import (
    build_cashflow_wide,
    compute_cfo_ni_ratio,
    compute_accruals,
    compute_composite_earnings_quality,
)
from utils.factor_analysis import (
    compute_ic_series,
    ic_summary,
    neutralize_factor,
    quintile_backtest,
)

print('✅ 导入成功')

## 1. 准备股票池和价格数据

In [ ]:
# ── 股票池：沪深 300 成分（快速验证先取前 80 只）
# 正式回测建议替换为 universe_at_date() 避免幸存者偏差
try:
    import akshare as ak
    hs300 = ak.index_stock_cons_weight_csindex(symbol="000300")
    symbols = hs300['成分券代码'].str.zfill(6).tolist()[:80]
    print(f'股票池: 沪深300成分 {len(symbols)} 只')
except Exception as e:
    print(f'akshare 拉取失败: {e}')
    # fallback: 用固定列表测试
    symbols = [
        '600519', '000858', '601318', '600036', '000333',
        '601166', '600887', '002594', '000651', '601012',
        '600900', '601288', '600276', '000568', '002415',
        '601398', '600104', '600585', '000002', '600030',
    ]
    print(f'使用 fallback 股票池: {len(symbols)} 只')

START = '2019-01-01'
END   = '2024-12-31'
date_range = pd.bdate_range('2020-01-01', END)  # 因子需要 2019 年数据预热
print(f'日期区间: {date_range[0].date()} ~ {date_range[-1].date()} | {len(date_range)} 个交易日')

## 2. 拉取现金流数据

In [ ]:
# 首次运行较慢（每只股票约 0.3s），有缓存后秒级完成
# 数据缓存在 data/raw/fundamentals/{symbol}_cashflow.parquet
print('正在拉取现金流量表数据（首次约 5~10 分钟）...')

cashflow = build_cashflow_wide(
    symbols=symbols,
    start='2019-01-01',
    end='2024-12-31',
    max_workers=6,
)

print('\n数据覆盖概况：')
for field, df in cashflow.items():
    coverage = df.notna().mean().mean()
    print(f'  {field:15s}: {df.shape} | 覆盖率 {coverage:.1%} | 时间 {df.index[0].date()} ~ {df.index[-1].date()}')

## 3. 计算子因子和合成因子

In [ ]:
# 子因子（日频宽表）
factor_cfo_ni    = compute_cfo_ni_ratio(cashflow, date_range)
factor_accruals  = compute_accruals(cashflow, date_range)
factor_composite = compute_composite_earnings_quality(cashflow, date_range)

print('子因子形状：')
print(f'  CFO/NI:    {factor_cfo_ni.shape} | 非空率 {factor_cfo_ni.notna().mean().mean():.1%}')
print(f'  Accruals:  {factor_accruals.shape} | 非空率 {factor_accruals.notna().mean().mean():.1%}')
print(f'  合成因子:  {factor_composite.shape} | 非空率 {factor_composite.notna().mean().mean():.1%}')

# 数据质量门
assert factor_composite.notna().mean().mean() > 0.3, '合成因子覆盖率过低，检查数据拉取'
print('\n✅ 数据质量检查通过')

In [ ]:
# 截面分布预览（取某日快照）
sample_date = pd.Timestamp('2023-06-30')
sample_snap = factor_composite.loc[:sample_date].iloc[-1].dropna()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, factor) in zip(axes, [
    ('CFO/NI', factor_cfo_ni),
    ('-Accruals/TA', factor_accruals),
    ('合成因子', factor_composite),
]):
    snap = factor.loc[:sample_date].iloc[-1].dropna()
    ax.hist(snap, bins=20, edgecolor='white', color='steelblue')
    ax.set_title(f'{name}\n截面分布（{sample_date.date()}）')
    ax.set_xlabel('因子值')
    ax.set_ylabel('频次')
plt.tight_layout()
plt.savefig('eq_factor_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('截面分布图已保存')

## 4. 准备前向收益

In [ ]:
from utils.data_loader import get_stock_history

# 拉取收盘价
print('拉取收盘价...')
price_frames = {}
for sym in symbols:
    try:
        df = get_stock_history(sym, start=START, end=END)
        if df is not None and not df.empty and 'close' in df.columns:
            price_frames[sym] = df['close']
    except Exception:
        pass

close_wide = pd.DataFrame(price_frames)
close_wide.index = pd.to_datetime(close_wide.index)
close_wide = close_wide.sort_index()
print(f'收盘价宽表: {close_wide.shape} | 时间: {close_wide.index[0].date()} ~ {close_wide.index[-1].date()}')

# 20 日前向收益（持仓 20 个交易日）
HOLD_DAYS = 20
fwd_ret = close_wide.pct_change(HOLD_DAYS).shift(-HOLD_DAYS)
print(f'前向收益（{HOLD_DAYS}日）: {fwd_ret.notna().mean().mean():.1%} 覆盖')

## 5. 单因子 IC 分析（未中性化）

In [ ]:
# 对齐日期
common_dates = factor_composite.index.intersection(fwd_ret.index)
factor_aligned = factor_composite.reindex(index=common_dates, columns=fwd_ret.columns)
fwd_aligned    = fwd_ret.reindex(index=common_dates, columns=fwd_ret.columns)

# Rank IC（Spearman 相关系数）
ic_raw = compute_ic_series(factor_aligned, fwd_aligned, method='spearman')
print('=== 未中性化 Rank IC ===' )
ic_summary(ic_raw, name='盈利质量合成因子（原始）')

In [ ]:
# IC 时间序列图
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ic_raw.plot(ax=axes[0], color='steelblue', alpha=0.7, label='月度 Rank IC')
ic_raw.rolling(6).mean().plot(ax=axes[0], color='darkblue', linewidth=2, label='6月滚动均值')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].axhline(0.02, color='green', linewidth=0.8, linestyle=':', label='IC=0.02')
axes[0].axhline(-0.02, color='red', linewidth=0.8, linestyle=':')
axes[0].set_title('盈利质量因子 — Rank IC 时序')
axes[0].legend()
axes[0].set_xlabel('')

ic_raw.cumsum().plot(ax=axes[1], color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('累积 IC')
axes[1].set_xlabel('日期')

plt.tight_layout()
plt.savefig('eq_ic_series.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 行业 + 市值中性化后复测

In [ ]:
# 构造行业/市值信息表（df_info 格式：date, symbol, industry, log_mktcap）
# 如果有 data/processed/stock_info.parquet，直接读取
import os
INFO_PATH = '../../../data/processed/stock_info.parquet'

if os.path.exists(INFO_PATH):
    df_info = pd.read_parquet(INFO_PATH)
    print(f'读取 stock_info: {df_info.shape}')
    
    factor_neutral = neutralize_factor(factor_aligned, df_info, n_sigma=3.0)
    ic_neutral = compute_ic_series(factor_neutral, fwd_aligned, method='spearman')
    print('\n=== 中性化后 Rank IC ===')
    ic_summary(ic_neutral, name='盈利质量合成因子（中性化）')
else:
    print('⚠️  stock_info.parquet 不存在，跳过中性化步骤')
    print('   运行 utils/fundamental_loader.py 生成或手动准备 df_info')
    factor_neutral = factor_aligned  # fallback: 用原始因子继续

## 7. 五分位分层回测

In [ ]:
# quintile_backtest: 返回每个分组的平均前向收益
qret = quintile_backtest(
    factor=factor_neutral,
    fwd_ret=fwd_aligned,
    n_quantiles=5,
)

print('各分位组平均前向收益（%）：')
print(qret.mean() * 100)
print(f'\n多空价差（Q5 - Q1）: {(qret["Q5"] - qret["Q1"]).mean() * 100:.3f}%/期')

In [ ]:
# 分层累积收益图
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 左图：各分位平均收益 bar
qret_mean = qret.mean() * 100
colors = ['#d32f2f', '#f57c00', '#fbc02d', '#388e3c', '#1565c0']
axes[0].bar(qret_mean.index, qret_mean.values, color=colors, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('各分位组平均持仓期收益')
axes[0].set_xlabel('因子分位（Q1=低质量, Q5=高质量）')
axes[0].set_ylabel('平均收益（%）')

# 右图：多空净值曲线
ls_ret = (qret['Q5'] - qret['Q1']).dropna()
ls_nav = (1 + ls_ret).cumprod()
ls_nav.plot(ax=axes[1], color='steelblue', linewidth=2)
axes[1].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('多空策略净值（Q5 - Q1）')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('净值')

plt.tight_layout()
plt.savefig('eq_quantile_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 子因子单独 IC 对比

In [ ]:
# 对比三个因子的 IC 统计
factor_dict = {
    'CFO/NI': factor_cfo_ni,
    '-Accruals/TA': factor_accruals,
    '合成因子': factor_composite,
}

print('=' * 60)
for name, f in factor_dict.items():
    f_aligned = f.reindex(index=common_dates, columns=fwd_ret.columns)
    ic_s = compute_ic_series(f_aligned, fwd_aligned, method='spearman')
    print(f'\n{name}')
    ic_summary(ic_s, name=name)

## 9. 与现有英雄因子的截面相关性

In [ ]:
from scipy.stats import spearmanr

# 加载现有因子（如果存在）
hero_factors = {}

try:
    from research.factors.quality.quality_factor import compute_quality_composite
    # quality_factor 需要 fundamental_loader 数据，这里仅示意结构
    print('quality_factor 模块加载成功（数据依赖需单独准备）')
    # hero_factors['quality'] = compute_quality_composite(...)  # 按需启用
except ImportError:
    print('quality_factor 不可用')

if hero_factors:
    print('\n=== 截面相关性（Spearman）===')
    new_factor = factor_composite
    for existing_name, existing_factor in hero_factors.items():
        corr_series = []
        for date in common_dates:
            if date not in existing_factor.index:
                continue
            f1 = new_factor.loc[date].dropna()
            f2 = existing_factor.loc[date].dropna()
            idx = f1.index.intersection(f2.index)
            if len(idx) > 30:
                corr_series.append(spearmanr(f1[idx], f2[idx])[0])
        if corr_series:
            print(f'  vs {existing_name}: 截面相关均值 = {np.mean(corr_series):.3f} | '
                  f'判断: {"⚠️ 相关性偏高" if abs(np.mean(corr_series)) > 0.3 else "✅ 增量信息有效"}')
else:
    print('\n⚠️  未找到现有英雄因子，跳过相关性检验')
    print('   请先运行各因子的数据准备脚本，再对比')

## 10. 总结

| 指标 | CFO/NI | -Accruals/TA | 合成因子 |
|------|--------|-------------|--------|
| Mean IC | - | - | - |
| ICIR | - | - | - |
| IC>0 比例 | - | - | - |
| Q5-Q1 多空 | - | - | - |

**前视偏差注意**：本回测使用 shift(1) 粗略处理，等价于滞后一个季度使用数据。
实际公告日比报告期末晚 1~3 个月，如需精确处理，替换为 announcement_date 时间戳。

**推进建议**：
- IC |t| > 2 且 ICIR > 0.3：进入多因子合成阶段
- 与现有 quality 因子相关性 < 0.3：增量有效，可加入 `ic_weighted_composite`
- 与现有 quality 因子相关性 > 0.5：考虑替换而非叠加